<a href="https://colab.research.google.com/github/NabilaAzzahra/rancangan-arsitektur-solusi-AI/blob/main/poc_rag_bank_nusantara_mandiri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PoC RAG Layanan Informasi Nasabah (PT Bank Nusantara Mandiri, fiktif)

Notebook ini menjalankan eksperimen Bab 5: **B0** (LLM tanpa RAG) melawan **R1** (RAG: bge-m3, ChromaDB, chunk 500 token, overlap 10%, K = 5, klasifikasi intent, ambang skor retrieval) pada 30 pertanyaan uji. Notebook ini menghasilkan jawaban dan lembar penilaian. Metrik Ragas dan ringkasan dihitung di notebook lanjutan (`poc_ragas_lanjutan.ipynb`).

**Langkah singkat**
1. Buat API key di Google AI Studio, lalu simpan di Colab: ikon kunci (Secrets) di kiri, nama `GOOGLE_API_KEY`, aktifkan *Notebook access*.
2. Unggah `poc_data.zip` ke panel File Colab (atau jalankan sel unggah di bawah).
3. Jalankan **Runtime > Run all**. Sel terakhir mengunduh `hasil_mentah.csv` dan `lembar_penilaian.csv`.
4. Isi kolom `nilai_B0` dan `nilai_R1` di `lembar_penilaian.csv` dengan **benar**, **sebagian**, atau **salah**.
5. Lanjutkan dengan `poc_ragas_lanjutan.ipynb` memakai `hasil_mentah.csv` tersebut.

Seluruh dokumen dan pertanyaan bersifat **fiktif**. Parameter pada sel konfigurasi ditetapkan sebelum pengukuran dan tidak diubah setelah melihat hasil.

In [1]:
!pip -q install google-genai sentence-transformers chromadb pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requi

In [2]:
import os, re, time, json, glob, zipfile, getpass, random
import numpy as np
import pandas as pd

# ---- Konfigurasi (ditetapkan sebelum pengukuran; jangan diubah setelah melihat hasil) ----
MODEL = "gemini-3.5-flash-lite"  # model pengganti gemini-2.5-flash (tidak lagi tersedia untuk akun baru)
TEMPERATURE = 0.2
THINKING = None                 # memakai bawaan model (3.5 Flash-Lite: thinking minimal); jangan diubah
K = 5
CHUNK_TOKENS = 500
OVERLAP = 0.10
SCORE_THRESHOLD = 0.30          # ambang skor kosinus retrieval (parameter penulis); konservatif karena topik di luar cakupan sudah disaring klasifikasi intent
REQUEST_SLEEP = 0.0             # naikkan (mis. 6) bila muncul galat 429 / kuota
DATA_DIR = "poc"
random.seed(0); np.random.seed(0)

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("GOOGLE_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = API_KEY
print("Konfigurasi siap:", MODEL, "K =", K, "ambang =", SCORE_THRESHOLD)
assert "2.5" not in MODEL, "Model 2.5 tidak lagi tersedia untuk akun baru; pakai gemini-3.5-flash-lite atau gemini-3.8-flash"

Konfigurasi siap: gemini-3.5-flash-lite K = 5 ambang = 0.3


In [3]:
# Unggah dan ekstrak data (lewati bila folder 'poc' sudah ada)
if not os.path.isdir(DATA_DIR):
    if not os.path.exists("poc_data.zip"):
        from google.colab import files
        files.upload()
    with zipfile.ZipFile("poc_data.zip") as z:
        z.extractall(".")
print(sorted(os.listdir(DATA_DIR)))
print(sorted(os.listdir(os.path.join(DATA_DIR, "docs"))))

['docs', 'pertanyaan_uji.csv']
['DOC-001_tarif_tabungan_kartu_debit.md', 'DOC-002_syarat_kredit.md', 'DOC-003_prosedur_kartu.md', 'DOC-004_faq.md']


## 1. Dokumen, chunking, dan indeks

In [4]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("BAAI/bge-m3")

def tok_len(text):
    try:
        return len(embedder.tokenizer.encode(text, add_special_tokens=False))
    except Exception:
        return int(len(text.split()) * 1.3)

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True, show_progress_bar=False).tolist()

def parse_doc(path):
    txt = open(path, encoding="utf-8").read()
    head, _, body = txt.partition("\n## ")
    meta = {}
    for line in head.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    sections = []
    for part in re.split(r"\n(?=## )", "## " + body):
        part = part.strip()
        if not part:
            continue
        title, _, text = part.partition("\n")
        sections.append((title[3:].strip(), text.strip()))
    return meta, sections

def split_long(text, max_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    if tok_len(text) <= max_tokens:
        return [text]
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        cur.append(s)
        if tok_len(" ".join(cur)) > max_tokens and len(cur) > 1:
            cur.pop()
            out.append(" ".join(cur))
            keep, n = [], 0
            for t in reversed(cur):
                n += tok_len(t)
                if n > max_tokens * overlap:
                    break
                keep.insert(0, t)
            cur = keep + [s]
    if cur:
        out.append(" ".join(cur))
    return out

CHUNKS = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "docs", "*.md"))):
    meta, secs = parse_doc(path)
    for title, text in secs:
        label = title.split(".")[0].strip()          # mis. "Pasal 2" atau "FAQ 1"
        for j, piece in enumerate(split_long(text)):
            cid = f"{meta['doc_id']}#{label}" + (f"-{j+1}" if j else "")
            header = f"[{meta['doc_id']} v{meta['versi']} | {meta['jenis']} | {label}] {title}"
            CHUNKS.append({"chunk_id": cid, "doc_id": meta["doc_id"], "versi": meta["versi"],
                           "jenis": meta["jenis"], "label": label, "text": header + "\n" + piece})
CHUNK_TEXT = {c["chunk_id"]: c["text"] for c in CHUNKS}
print(len(CHUNKS), "chunk;", "maks token per chunk =", max(tok_len(c["text"]) for c in CHUNKS))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

23 chunk; maks token per chunk = 89


In [5]:
import chromadb
chroma = chromadb.Client()
col = chroma.create_collection("poc", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in CHUNKS],
        documents=[c["text"] for c in CHUNKS],
        embeddings=embed([c["text"] for c in CHUNKS]),
        metadatas=[{"doc_id": c["doc_id"], "jenis": c["jenis"], "label": c["label"]} for c in CHUNKS])

def retrieve(q, k=K):
    r = col.query(query_embeddings=embed([q]), n_results=k)
    ids = r["ids"][0]; docs = r["documents"][0]
    scores = [1.0 - d for d in r["distances"][0]]      # jarak kosinus -> kemiripan
    return docs, ids, scores
print("Indeks siap:", col.count(), "chunk")

Indeks siap: 23 chunk


## 2. LLM, prompt, dan dua konfigurasi

In [6]:
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)

USE_TEMP = True   # akan dimatikan otomatis bila model menolak parameter temperature

def gemini(prompt, system=None, temperature=TEMPERATURE, retries=6):
    global USE_TEMP
    for a in range(retries):
        kw = {"system_instruction": system}
        if USE_TEMP: kw["temperature"] = temperature
        cfg = types.GenerateContentConfig(**kw)
        try:
            if REQUEST_SLEEP: time.sleep(REQUEST_SLEEP)
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            u = r.usage_metadata
            tin = int(getattr(u, "prompt_token_count", 0) or 0)
            tout = int((getattr(u, "candidates_token_count", 0) or 0) + (getattr(u, "thoughts_token_count", 0) or 0))
            return (r.text or "").strip(), tin, tout
        except Exception as e:
            msg = str(e)
            if "404" in msg or "NOT_FOUND" in msg:
                print("MODEL TIDAK DITEMUKAN:", MODEL, "-", msg[:200])
                try:
                    print("Model yang tersedia untuk key ini:")
                    for m in client.models.list():
                        acts = getattr(m, "supported_actions", None) or []
                        if "generateContent" in acts: print("  ", m.name)
                except Exception as e2:
                    print("(gagal menampilkan daftar model)", str(e2)[:100])
                raise RuntimeError(f"Ubah MODEL di sel konfigurasi ke salah satu model di atas. ({MODEL} tidak tersedia)")
            if USE_TEMP and "temperature" in msg.lower():
                USE_TEMP = False; print("Model menolak temperature; dijalankan tanpa temperature (catat di 5.5)."); continue
            print("coba ulang", a + 1, "->", msg[:140]); time.sleep(3 * 2 ** a)
    raise RuntimeError("Panggilan Gemini gagal berulang")

INTENT_SYS = '''Klasifikasikan pesan nasabah PT Bank Nusantara Mandiri ke tepat satu label:
- produk: pertanyaan informasi produk, tarif, syarat, prosedur, atau jam layanan (tabungan, kartu debit, KTA, KPR, kartu kredit).
- sensitif: keluhan, sengketa transaksi, dugaan penipuan, permintaan saldo atau data rekening, atau permintaan data pribadi.
- di_luar: selain itu (topik umum, investasi, bank lain, hiburan).
Jawab hanya dengan satu kata: produk, sensitif, atau di_luar.'''

GEN_SYS = '''Kamu asisten virtual PT Bank Nusantara Mandiri (fiktif). Jawab pertanyaan nasabah dalam bahasa Indonesia yang ringkas dan sopan.
Aturan:
1. Jawab HANYA berdasarkan konteks yang diberikan. Jangan memakai pengetahuan lain.
2. Bila konteks memuat dokumen resmi dan FAQ yang bertentangan, gunakan dokumen resmi.
3. Cantumkan sumber di akhir jawaban dengan format [DOC-xxx Pasal n] atau [DOC-xxx FAQ n].
4. Jangan memberi rekomendasi keuangan atau keputusan kredit.
5. Bila konteks tidak cukup untuk menjawab, balas persis: TIDAK_TAHU'''

B0_SYS = '''Kamu asisten virtual PT Bank Nusantara Mandiri. Jawab pertanyaan nasabah dalam bahasa Indonesia yang ringkas dan sopan.'''

MSG_ESKALASI = "Pertanyaan ini perlu ditangani petugas kami. Percakapan Anda akan dialihkan ke petugas."
MSG_TOLAK = "Maaf, saya hanya dapat membantu informasi produk dan layanan PT Bank Nusantara Mandiri. Silakan hubungi kanal resmi bank."
MSG_TIDAK_TAHU = "Maaf, saya belum menemukan informasi yang cukup untuk menjawab. Percakapan Anda akan dialihkan ke petugas."

def classify(q):
    out, i, o = gemini(q, system=INTENT_SYS, temperature=0.0)
    low = out.lower()
    for lab in ("sensitif", "di_luar", "produk"):
        if lab in low:
            return lab, i, o
    return "produk", i, o

def build_prompt(q, ctx):
    return "Konteks:\n" + "\n\n".join(ctx) + f"\n\nPertanyaan nasabah: {q}\nJawaban:"

def run_b0(q):
    t0 = time.perf_counter()
    ans, i, o = gemini(q, system=B0_SYS)
    return {"jawaban": ans, "aksi": "jawab", "intent": "-", "konteks_ids": "", "skor_top1": np.nan,
            "latensi_s": time.perf_counter() - t0, "token_in": i, "token_out": o}

def run_r1(q):
    t0 = time.perf_counter(); tin = tout = 0
    lab, i, o = classify(q); tin += i; tout += o
    ids, scores, ctx = [], [], []
    if lab == "sensitif":
        aksi, ans = "eskalasi", MSG_ESKALASI
    elif lab == "di_luar":
        aksi, ans = "tolak", MSG_TOLAK
    else:
        ctx, ids, scores = retrieve(q)
        if not ctx or scores[0] < SCORE_THRESHOLD:
            aksi, ans = "eskalasi", MSG_TIDAK_TAHU
        else:
            ans, i, o = gemini(build_prompt(q, ctx), system=GEN_SYS); tin += i; tout += o
            if "TIDAK_TAHU" in ans:
                aksi, ans = "eskalasi", MSG_TIDAK_TAHU
            else:
                aksi = "jawab"
    return {"jawaban": ans, "aksi": aksi, "intent": lab, "konteks_ids": "|".join(ids),
            "skor_top1": scores[0] if scores else np.nan, "konteks_teks": ctx,
            "latensi_s": time.perf_counter() - t0, "token_in": tin, "token_out": tout}
print("Fungsi siap")

Fungsi siap


## 3. Menjalankan 30 pertanyaan uji

In [7]:
Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
assert len(Q) == 30, len(Q)
rows = []
for n, r in Q.iterrows():
    b0 = run_b0(r["pertanyaan"]); r1 = run_r1(r["pertanyaan"])
    rows.append({"id": r["id"], "tipe": r["tipe"], "aksi_diharapkan": r["aksi_diharapkan"], "B0": b0, "R1": r1})
    print(f'{r["id"]} {r["tipe"][:8]:8} R1={r1["aksi"]:8} {r1["latensi_s"]:.1f}s  skor={r1["skor_top1"]}')

flat = []
for x in rows:
    for cfg in ("B0", "R1"):
        d = {k: v for k, v in x[cfg].items() if k != "konteks_teks"}
        d.update({"id": x["id"], "tipe": x["tipe"], "aksi_diharapkan": x["aksi_diharapkan"], "konfigurasi": cfg})
        flat.append(d)
RAW = pd.DataFrame(flat)
RAW["token_total"] = RAW["token_in"] + RAW["token_out"]
RAW.to_csv("hasil_mentah.csv", index=False, encoding="utf-8-sig")
print("Disimpan: hasil_mentah.csv")

Q01 dalam_ca R1=jawab    2.0s  skor=0.7846357822418213
Q02 dalam_ca R1=jawab    1.7s  skor=0.788307249546051
Q03 dalam_ca R1=jawab    1.4s  skor=0.8219337463378906
Q04 dalam_ca R1=jawab    1.5s  skor=0.6411563158035278
Q05 dalam_ca R1=jawab    1.6s  skor=0.7777507901191711
coba ulang 1 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 2 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 3 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
Q06 dalam_ca R1=jawab    23.3s  skor=0.7534356117248535
coba ulang 1 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 2 -> 429 RESOURCE_EXHAUSTED. {'

In [8]:
# Lembar penilaian manusia (isi kolom nilai_B0 dan nilai_R1: benar / sebagian / salah)
J = Q[["id", "tipe", "pertanyaan", "jawaban_referensi"]].copy()
rr = RAW.set_index(["id", "konfigurasi"])
J["jawaban_B0"] = [rr.loc[(i, "B0"), "jawaban"] for i in J.id]
J["jawaban_R1"] = [rr.loc[(i, "R1"), "jawaban"] for i in J.id]
J["aksi_R1"] = [rr.loc[(i, "R1"), "aksi"] for i in J.id]
J["nilai_B0"] = ""; J["nilai_R1"] = ""
J.to_csv("lembar_penilaian.csv", index=False, encoding="utf-8-sig")
print("lembar_penilaian.csv dibuat. Hanya pertanyaan 'dalam_cakupan' yang perlu dinilai.")
try:
    from google.colab import files
    files.download("hasil_mentah.csv"); files.download("lembar_penilaian.csv")
except Exception:
    print("Unduh hasil_mentah.csv dan lembar_penilaian.csv dari panel File.")

lembar_penilaian.csv dibuat. Hanya pertanyaan 'dalam_cakupan' yang perlu dinilai.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>